# Introduction to the USF Seismology Computer Network
**Course support document (Computational Seismology)**  
_Last updated: 2026-02-25_

## Executive summary
The USF Seismology Group runs a **hybrid Linux/macOS research network** designed for (1) shared access to large datasets, (2) reproducible analysis workflows, and (3) resilient backups.

- **newton (2013, group-funded)** is the **primary group server** and the long-term destination for shared datasets, backups, and configuration management (Ansible). It currently needs a clean-up to free space for broader user backups.
- **hal9000 (2018, personally-funded)** is primarily **Glenn’s personal workstation/server** hosting datasets that matter to Glenn. It is also being used as the **course data share** right now because it is already organized and easier to operate (ZFS), while newton is mid-transition.
- **Linux desktops (“thin clients”, 2013)** provide local compute for students/faculty in the lab. They mount shared storage primarily via **NFS**.
- **macOS laptops/desktops** (and optionally Windows) access shared datasets via **Samba/SMB**.
- In the longer term, the intended “default” for the group is: **newton as primary**, with hal9000 serving Glenn’s personal workflows plus a convenience share for teaching when needed.


## 1) Network overview and roles (corrected)

### 1.1 Key machines and what they are “for”

#### **newton — group server (primary, 2013, group funds)**
- Rack-mounted server purchased for the Seismology Group (2013).
- Storage: **~37 TB** (legacy configuration: **16-disk JBOD**, **MegaRAID**, **XFS**).
- Current role (group reality): **primary group server** (shared datasets + planned backups + Ansible control).
- Practical constraint right now: storage is close to full, so large “rarely accessed” datasets need to move to offline storage before newton can become the default for broad automated user backups.

#### **hal9000 — personal server/workstation (2018, personal funds)**
- Tower workstation/server purchased by Glenn (2018).
- Storage: **ZFS** (primary pool + backup pool), enabling robust snapshots and easy rollback.
- Current role (Glenn reality): **primary home for Glenn’s important working datasets**.
- Current course role: used as the **class SMB share** because it is already organized and maintained, and because newton is mid-cleanup.

#### **Linux thin-client desktops (2013 lab fleet)**
- Older desktops (i7 class CPUs, **16 GB RAM**) used as lab workstations.
- Good for Python/ObsPy analysis, notebooks, and modest compute tasks.
- Primary share mechanism: **NFS mounts** from Linux servers.

#### **macOS laptops/desktops**
- Students/faculty often work on Macs.
- Data access: **SMB/Samba** mounts from Linux servers.
- Analysis: Python/Conda + ObsPy + notebooks.

#### **Windows (optional / not officially supported)**
- SMB/Windows file sharing usually works fine for mounting shared data.
- Windows is not Unix-based, so some shell workflows differ; course support focuses on macOS + Linux.


## 2) Network diagram (server roles and access paths)

The diagram below reflects the intended **group view** (newton primary) *and* the current **teaching reality** (hal9000 used for class share).

```text
                               ┌───────────────────────────────┐
                               │        Campus Network         │
                               │          (LAN/VLAN)           │
                               └───────────────┬───────────────┘
                                               │
               ┌───────────────────────────────┼───────────────────────────────┐
               │                               │                               │
               ▼                               ▼                               ▼
┌─────────────────────────-─┐      ┌──────────────────────────┐      ┌──────────────────────────┐
│         hal9000           │      │          newton          │      │   Linux Thin Clients     │
│ (2018, personal funds)    │      │ (2013, group funds)      │      │ (2013 lab desktops)      │
│                           │      │                          │      │                          │
│ Glenn’s primary data      │      │ Group primary server     │      │ Local compute + notebooks│
│ + course SMB share        │      │ + planned backups        │      │ mounts via NFS           │
│                           │      │ + Ansible control host   │      │                          │
│ ZFS primary pool          │      │ 16-disk JBOD             │      │ Python / ObsPy / Jupyter │
│ ZFS backup pool (OWC)     │      │ MegaRAID + XFS           │      └──────────────┬───────────┘
│ Snapshots (sanoid)        │      │ ~37 TB (nearly full)     │                     │
└──────────────┬─────────-──┘      └──────────────┬───────────┘                     │
               │                                  │                                 │
               │ SMB share (classdata)            │ NFS shares (Linux↔Linux)        │
               ▼                                  ▼                                 ▼
   ┌──────────────────────-─┐         ┌──────────────────────-─┐        ┌──────────────────────────┐
   │ Student/faculty Macs   │         │ Linux desktops/lab     │        │ Local analysis           │
   │ Finder mount           │         │ NFS mount paths        │        │ environment              │
   │ /Volumes/classdata     │         │ (/mnt/... etc.)        │        │ (Python/ObsPy/Notebooks) │
   └──────────────────────-─┘         └─────────────────-──────┘        └──────────────────────────┘

   ┌──────────────────────-─┐
   │ Student Windows (opt.) │
   │ Mapped drive (e.g. Z:) │
   └─────────────────────-──┘

                          ┌─────────────────────────────────────────────────────┐
                          │ Configuration Management (Linux network)            │
                          │ - newton runs Ansible to sync accounts/configs      │
                          │ - goal: consistent users + tools across lab Linux   │
                          └─────────────────────────────────────────────────────┘
```

**Interpretation:**
- **Group default (target):** newton is the primary shared server, plus configuration management.
- **Current course workaround:** hal9000 provides the easiest, safest SMB dataset share today.
- **Glenn’s workflow:** hal9000 is “primary for Glenn”; **backs up elsewhere**.


## 3) Data lifecycle (how data moves through the system)

```text
┌───────────────────────────────┐
│ 1) Acquisition / Ingest        │
│   • Field deployments          │
│   • Observatory downloads      │
│   • External archives          │
│   • Simulations                │
└───────────────┬───────────────┘
                │
                ▼
┌───────────────────────────────┐
│ 2) Active Working Data (HOT)   │
│   hal9000 (Glenn)              │
│   • active processing          │
│   • analysis notebooks         │
│   • class datasets (current)   │
└───────────────┬───────────────┘
                │  ZFS snapshots (version history)
                ▼
┌───────────────────────────────┐
│ 3) Local Backup / Fast Restore │
│   hal9000 backup pool (OWC)    │
│   • quick rollback             │
│   • protects against mistakes  │
└───────────────┬───────────────┘
                │ nightly mirrors (as configured)
                ▼
┌───────────────────────────────┐
│ 4) Group Archive / Shared      │
│   newton (group primary)       │
│   • shared group datasets      │
│   • longer-term retention      │
│   • future user backups        │
└───────────────┬───────────────┘
                │ periodic migration
                ▼
┌───────────────────────────────┐
│ 5) Offline / Cold Archive      │
│   external drives (powered off)│
│   • rarely accessed datasets   │
│   • ransomware-resistant copy  │
└───────────────────────────────┘
```


## 4) Backup strategy (what protects you from what)

### 4.1 Current backup strategy for hal9000 (Glenn’s primary data)
- **ZFS snapshots** on the primary pool via **sanoid**
- **rsync mirror** of selected directories (currently `/data`) to a **local ZFS backup pool**
- **future / additional mirrors** planned (e.g., also sync to newton; expand beyond `/data` to `/work` and possibly `/home`)

### 4.2 Diagram: backup layers and recovery paths

```text
                       ┌──────────────────────────────┐
                       │           hal9000            │
                       │   ZFS Primary Pool (HOT)     │
                       │  (Glenn’s primary datasets)  │
                       └──────────────┬───────────────┘
                                      │
                      sanoid snapshots│  (rollback/versioning)
                                      ▼
                       ┌──────────────────────────────┐
                       │   Snapshot History (ZFS)     │
                       │  daily/weekly/monthly policy │
                       └──────────────┬───────────────┘
                                      │
                         rsync mirror │  (nightly)
                                      ▼
                       ┌──────────────────────────────┐
                       │ hal9000 ZFS Backup Pool (OWC) │
                       │  fast restore + extra copy     │
                       └──────────────┬───────────────┘
                                      │
                           future sync│  (planned/periodic)
                                      ▼
                       ┌──────────────────────────────┐
                       │            newton             │
                       │  group server (WARM archive)  │
                       │  independent hardware         │
                       └──────────────┬───────────────┘
                                      │
                             migration│
                                      ▼
                       ┌──────────────────────────────┐
                       │   Offline / Cold Archive      │
                       │  powered-off external drives  │
                       └──────────────────────────────┘
```

### 4.3 Quick recovery guide
- **Accidental delete / overwrite:** rollback a ZFS snapshot (seconds–minutes)
- **Drive failure on hal9000:** restore from hal9000 backup pool (minutes–hours)
- **Total hal9000 failure:** restore from newton mirror (hours)
- **Worst case (ransomware / catastrophic):** restore from offline archive (days)


## 5) Data sharing in the course (why SMB, why hal9000)

For this course (Spring 2026), we are **not copying datasets to a special “class folder”**. Instead, we mount a **read-only SMB share** exported from hal9000.

### Why this approach?
- Minimal duplication of large datasets
- Students can browse and copy only what they need
- Read-only access prevents accidental damage
- Works on macOS (primary supported student platform) and usually on Windows

### Important note
In the long term, the group intention is for **newton** to be the primary data-sharing and backup hub. hal9000 is being used for teaching right now because it is already maintained and easier to operate safely.


## 6) Student instructions — macOS SMB mount (supported)

### Connect in Finder
1. Open **Finder**
2. Press **⌘K** (or Finder menu: **Go → Connect to Server…**)
3. Enter:
   - `smb://10.246.31.49/classdata`
4. Click **Connect**

### What you should see
- A new Finder location named **classdata**
- It will also appear in Terminal at:
  - `/Volumes/classdata`

### Read-only policy
You have **READ-ONLY** access. You cannot damage the source data on the server.


## 7) Student instructions — Windows SMB mount (one consistent method)

We will use **File Explorer → Map network drive** (this is consistent across Windows versions).

### Steps
1. Open **File Explorer**
2. Right-click **This PC**
3. Click **Map network drive…**
4. Choose a drive letter (we’ll standardize on **Z:** if available)
5. In the “Folder” box, enter:
   - `\\10.246.31.49\classdata`
6. Check:
   - ✅ **Reconnect at sign-in**
7. Click **Finish**

### What you should see in Windows
- In **File Explorer** under **This PC**, you’ll see a new drive:
  - **Z: (\\10.246.31.49\classdata)**
- In the Windows shell (PowerShell or Command Prompt), it appears as:
  - `Z:\Pinatubo\...`
  - Example: `Z:\Pinatubo\WAVEFORMS\199112\9112010B.DMX`

### Read-only policy
You have **READ-ONLY** access. You cannot modify or delete the server-side data.


## 8) Python: writing platform-independent paths (Windows/macOS/Linux)

### 8.1 Detect the OS
Use `platform.system()`:

```python
import platform
platform.system()
```
Typical returns:
- `"Darwin"` (macOS)
- `"Linux"`
- `"Windows"`

### 8.2 A generic “mount root” helper
This example assumes:
- macOS mount point: `/Volumes/classdata`
- Windows mapped drive: `Z:`
- Linux (if used): `/mnt/classdata` (adjust if your Linux mount differs)

```python
from pathlib import Path
import platform

def classdata_root() -> Path:
    sys = platform.system()
    if sys == "Darwin":
        return Path("/Volumes/classdata")
    elif sys == "Windows":
        return Path("Z:/")
    elif sys == "Linux":
        return Path("/mnt/classdata")
    else:
        raise RuntimeError(f"Unsupported OS: {sys}")

root = classdata_root()
print("Using classdata root:", root)
```

### 8.3 Generic ObsPy example
```python
from obspy import read
from pathlib import Path
import platform

def classdata_root() -> Path:
    sys = platform.system()
    if sys == "Darwin":
        return Path("/Volumes/classdata")
    elif sys == "Windows":
        return Path("Z:/")
    elif sys == "Linux":
        return Path("/mnt/classdata")
    else:
        raise RuntimeError(f"Unsupported OS: {sys}")

p = classdata_root() / "Pinatubo" / "WAVEFORMS" / "199112" / "9112010B.DMX"

st = read(str(p))
print(st)
st.plot()
```


## 9) Exercise: copying a dataset folder locally (recommended approach)

For most assignments, you should **copy only the subset you need** from the read-only share to your local machine (e.g., to work offline or improve performance).

### Recommended method (macOS/Linux): `rsync` from the mounted share
Choose a dataset folder and copy it into a local working directory.

#### Example (macOS)
```bash
mkdir -p ~/classdata_local/Pinatubo_subset

rsync -avh --progress   "/Volumes/classdata/Pinatubo/WAVEFORMS/199112/"   "~/classdata_local/Pinatubo_subset/199112/"
```

#### Example (Linux)
```bash
mkdir -p ~/classdata_local/Pinatubo_subset

rsync -avh --progress   "/mnt/classdata/Pinatubo/WAVEFORMS/199112/"   "~/classdata_local/Pinatubo_subset/199112/"
```

### Notes
- `rsync` preserves timestamps and is restartable.
- It does not require user accounts on the server **if** you can mount the SMB share.
- You are copying **from the mounted share** to your local disk, not logging into the server.


## 10) Preventing and cleaning up macOS metadata files (.DS_Store and ._*)

### 10.1 Linux cleanup command (run on servers or Linux workstations)
**Dry run first** (prints what would be deleted):
```bash
find /path/to/check -name '.DS_Store' -o -name '._*' -print
```

**Delete** (safe for typical data directories):
```bash
find /path/to/check \( -name '.DS_Store' -o -name '._*' \) -type f -print -delete
```

If you want to do it across the entire shared data tree on hal9000:
```bash
find /mnt/RAIDZ/share/data \( -name '.DS_Store' -o -name '._*' \) -type f -print -delete
```

### 10.2 macOS: reduce creation of .DS_Store on network shares
On each Mac, you can disable `.DS_Store` creation on **network volumes**:
```bash
defaults write com.apple.desktopservices DSDontWriteNetworkStores -bool TRUE
killall Finder
```

(Optional) also disable on **USB/external drives**:
```bash
defaults write com.apple.desktopservices DSDontWriteUSBStores -bool TRUE
killall Finder
```

### 10.3 Samba-side mitigation (server)
On hal9000 we use Samba options that help store mac metadata in xattrs instead of littering AppleDouble files:
- `vfs objects = catia fruit streams_xattr`
- `fruit:metadata = stream`
- `fruit:resource = stream`
- `ea support = yes`


## 11) hal9000: implemented snapshot + rsync backup (what we did)

### 11.1 Snapshotting with sanoid
- `sanoid` is configured to snapshot **entire pools**, not only `/data`.
- This is intentional: it provides protection for everything on the pool, and later we can expand rsync to include additional paths (e.g. `/work`, and possibly `/home`).

### 11.2 Rsync mirror to ZFS backup pool
We mirror:
- source: `/mnt/RAIDZ/share/data/`
- destination: `/ZPOOL_BACKUP/data/`

We exclude Apple metadata litter:
- `.DS_Store`
- `._*`

### 11.3 Cron schedule (current)
- sanoid runs periodically for snapshot/pruning
- rsync runs nightly
- zpool scrub runs monthly (each pool)

(See the server’s root crontab for the exact entries.)


## 12) Cluster and HPC (separate topic, not the focus of this course)

### 12.1 “Future local cluster” goal
We hope to build a small Linux cluster from:
- lab desktops (“thin clients”)
- surplus computers adopted from other groups (priority: newer than 2018 and **≥32 GB RAM**)

This will support parallel processing of large waveform archives and machine learning workflows.

### 12.2 CIRCE (USF IT research cluster)
USF IT maintains **CIRCE**, a powerful research HPC system.
- Access typically involves job queues.
- To get guaranteed capacity, groups usually need to purchase nodes (**~$30–50k minimum investment**).
- Because of the queuing model and procurement realities, **CIRCE is not covered in this course**, and most of us do not use it regularly.
